In [ ]:
!pip install optuna
import json
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch.nn as nn
from transformers import AutoTokenizer
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score, precision_score, recall_score
from typing import Optional
from huggingface_hub import hf_hub_download
import optuna
import pandas as pd
from google.colab import files  # Only if you want to auto-download

In [ ]:
def macro_metrics(
    y_true,
    y_pred,
    labels=(-1, 0, 1)
):
    """
    Macro class-wise metrics:
    - Compute per-aspect multiclass F1/Precision/Recall
    - Average across aspects

    y_true, y_pred: (N, S)
    """

    precisions, recalls, f1s = [], [], []

    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        precisions.append(
            precision_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        recalls.append(
            recall_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        f1s.append(
            f1_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )

    return {
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
    }
def sample_metrics(
    y_true,
    y_pred,
    labels=(-1, 0, 1)
):
    """
    Sample-based metrics:
    - Compute metrics per sample across all aspects
    - Then average over samples
    """

    precisions, recalls, f1s = [], [], []

    for i in range(y_true.shape[0]):
        yt = y_true[i]
        yp = y_pred[i]

        precisions.append(
            precision_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        recalls.append(
            recall_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        f1s.append(
            f1_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )

    return {
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
    }


In [ ]:
class EvalDataset(Dataset):
    def __init__(self, data, tokenizer, top_dim, sub_dim, max_len=256, label_map=None):
        self.data = data
        self.tokenizer = tokenizer
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.max_len = max_len
        self.label_map = label_map # Store label_map

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}

        top = torch.tensor(item["top_cluster_ids"], dtype=torch.float32)
        if top.ndim == 0:  # single int label
            top = F.one_hot(top.long(), num_classes=self.top_dim).float()

        sub_ids = torch.tensor(item["sub_cluster_ids"], dtype=torch.float32)

        raw_sentiments = item.get("sentiments", {})
        sentiments = {}
        for k, v in raw_sentiments.items():
            mapped_v = v
            if self.label_map:
                mapped_v = self.label_map.get(v)

            if mapped_v is not None: # Only include if a valid (non-None) value is found/mapped
                sentiments[int(k)] = mapped_v

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "top_labels": top,
            "sub_labels": sub_ids,
            "sentiments": sentiments,
        }

    @staticmethod
    def collate_fn(batch):
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
            "top_labels": torch.stack([b["top_labels"] for b in batch]),
            "sub_labels": torch.stack([b["sub_labels"] for b in batch]),
            "sentiments": [b["sentiments"] for b in batch],
        }

In [161]:
def load_tokenizer_and_model_from_hf():
    repo_id = "Faisal191/aspect-classifier"
    encoder_subfolder = "Domain_trained_encoder"

    onnx_file = "HABSA/Habsa_v6_fp32.onnx"

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        repo_id,
        subfolder=encoder_subfolder)
    # Download ONNX graph
    onnx_path = hf_hub_download(
        repo_id=repo_id,
        filename=onnx_file,
        repo_type="model")

    return tokenizer, onnx_path

tokenizer, onnx_path = load_tokenizer_and_model_from_hf()


In [162]:
top_logits, sub_logits, sent_logits = [], [], []

In [163]:
top_logits = np.load("/content/top_logits.npy")
sub_logits = np.load("/content/sub_logits.npy")
sent_logits = np.load("/content/sent_logits.npy")

In [164]:
print(top_logits.shape)
print(sub_logits.shape)
print(sent_logits.shape)

(793, 9)
(793, 29)
(793, 29, 2)


In [165]:
outputs = {"top_logits": top_logits,
           "sub_logits": sub_logits,
           "sent_logits": sent_logits}

In [166]:
def sentiment_from_logits(sent_logits, pos_margin, neg_margin):
    delta = sent_logits[..., 1] - sent_logits[..., 0]

    sent = torch.full_like(delta, -1, dtype=torch.long)  # -1 = abstain
    sent[delta > pos_margin] = 1
    sent[delta < -neg_margin] = 0
    return sent


In [167]:
def inference(top_to_sub_dense, top_logits, sub_logits, sent_logits, top_thresholds,
              sub_thresholds, pos_margin, neg_margin):
    top_logits = torch.from_numpy(top_logits)
    sub_logits = torch.from_numpy(sub_logits)
    sent_logits = torch.from_numpy(sent_logits)
    p_top = torch.sigmoid(top_logits)        # (B, T)
    p_sub = torch.sigmoid(sub_logits)        # (B, S)
    top_thr = torch.tensor(top_thresholds).unsqueeze(0)   # (1, T)
    sub_thr = torch.tensor(sub_thresholds).unsqueeze(0)   # (1, S)
    p_top_bin = (p_top > top_thr).to(torch.float32)
    top_to_sub_dense = torch.tensor(top_to_sub_dense)
    mask = torch.matmul(p_top_bin, top_to_sub_dense)  # (B, S)
    mask = mask.clamp(0.0, 1.0)
    no_top = (p_top_bin.sum(dim=1, keepdim=True) == 0).to(torch.float32)  # (B, 1)
    mask = mask + no_top * (1.0 - mask)
    p_sub_masked = p_sub * mask
    pred_sub_bin = (p_sub_masked > sub_thr).to(torch.float32)
    sent_preds = sentiment_from_logits(sent_logits, pos_margin, neg_margin)

    return pred_sub_bin.cpu().numpy(), sent_preds.cpu().numpy()

In [168]:
def eval(pred_sub_bin, sent_preds, y_sub, sent_labels):
    f1_sub = f1_score(y_sub, pred_sub_bin, average="macro", zero_division=0)
    recall_sub = recall_score(y_sub, pred_sub_bin, average="macro", zero_division=0)
    precision_sub = precision_score(y_sub, pred_sub_bin, average="macro", zero_division=0)

    sent_preds_flat = sent_preds.flatten()
    sent_labels_flat = sent_labels.flatten()

    mask = (sent_labels_flat != -1) & (sent_preds_flat != -1)

    f1_sent = f1_score(sent_labels_flat[mask], sent_preds_flat[mask], average="macro", zero_division=0)
    rec_sent = recall_score(sent_labels_flat[mask], sent_preds_flat[mask], average="macro", zero_division=0)
    prec_sent = precision_score(sent_labels_flat[mask], sent_preds_flat[mask], average="macro", zero_division=0)
    joint_preds = sent_preds.copy()
    joint_preds[(pred_sub_bin == 0)] = -1
    hard_class = macro_metrics(sent_labels, joint_preds)
    """joint_precision": hard_class["precision"], "joint_recall": hard_class["recall"], "joint_f1": hard_class["f1"], "sentiment_f1": f1_sent, "sentiment_precision": prec_sent, "sentiment_recall": rec_sent , "sub_f1": f1_sub,"""
    return { "aspect_precision": precision_sub, "aspect_recall": recall_sub}



In [169]:
CFG = {
    "hierarchical_json": r"/content/final_aspa_data_hierarchical_with_sentiments_temp_v6.json",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


In [170]:
def run(cfg):
    DEVICE = torch.device(cfg["device"])
    sample_json = json.load(open(cfg["hierarchical_json"], "r"))
    eval_dataset = EvalDataset(sample_json, tokenizer, top_dim=9, sub_dim=29, max_len=256, label_map={0: 0, 1: 1})
    valid_indices = np.load("/content/valid_indices (2).npy")
    top_to_sub_dense = np.load("/content/top_to_sub_dense (1).npy")
    eval_ds = torch.utils.data.Subset(eval_dataset, valid_indices)
    eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False, collate_fn=EvalDataset.collate_fn)

    y_sub, sentiments = [], []
    for batch in tqdm(eval_loader):
        y_sub.append(batch["sub_labels"].cpu().numpy())
        sentiments += batch["sentiments"]
    y_sub = np.concatenate(y_sub)

    len_eval_data = len(eval_loader.dataset)
    sent_labels = np.full((len_eval_data, 29), -1.0)

    for i, row_i in enumerate(sentiments):
      aspect, sentiments = list(row_i.keys()), list(row_i.values())
      sent_labels[i, aspect] = sentiments
    return y_sub, sent_labels, top_to_sub_dense

In [ ]:
y_sub, sent_labels, top_to_sub_dense = run(CFG)

100%|██████████| 13/13 [00:00<00:00, 17.89it/s]


In [174]:
FIXED_TOP_THRESHOLDS = [0.5] * 9

In [183]:
study1 = optuna.create_study(
    directions=[
        "maximize",  # aspect precision
        "maximize",  # aspect recall
    ],

    sampler = optuna.samplers.TPESampler(
        n_startup_trials=100,      # pure random trials
        n_ei_candidates=64,       # explore more candidates
        multivariate=True,
        group=True,
        seed=42)
)

def optuna_objective(trial):
    top_thresholds = [
            trial.suggest_float(f"top_thr_{j}", 0.15, 0.5)
            for j in range(9)
        ]

    sub_thresholds = [
        trial.suggest_float(f"sub_thr_{j}", 0.15, 0.5)
        for j in range(29)]

    pred_sub_bin, sent_preds = inference(
        top_to_sub_dense,
        outputs["top_logits"],
        outputs["sub_logits"],
        outputs["sent_logits"],
        top_thresholds,
        sub_thresholds,
        pos_margin=0,
        neg_margin=0)

    metrics = eval(pred_sub_bin, sent_preds, y_sub, sent_labels)
    if metrics["aspect_recall"] < 0.75:
        return metrics["aspect_recall"]-1, metrics["aspect_recall"]-1

    return (
        metrics["aspect_precision"],
        metrics["aspect_recall"])
study1.optimize(optuna_objective, n_trials=500)


/usr/local/lib/python3.12/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2025-12-30 15:08:33,829] A new study created in memory with name: no-name-5f6c70fb-5c0b-4e7f-a527-ccc95a6dc0a5
[I 2025-12-30 15:08:34,195] Trial 0 finished with values: [-0.2504587263679492, -0.2504587263679492] and parameters: {'top_thr_0': 0.2810890415965769, 'top_thr_1': 0.4827500072434706, 'top_thr_2': 0.4061978796339918, 'top_thr_3': 0.35953046946896283, 'top_thr_4': 0.20460652415485278, 'top_thr_5': 0.20459808211767092, 'top_thr_6': 0.1703292642588698, 'top_thr_7': 0.4531616510212273, 'top_thr_8': 0.36039025411012304, 'sub_thr_0': 0.3978254022286159, 'sub_thr_1': 0.15720457300353086, 'sub_th

In [184]:
pareto_trials = study1.best_trials

In [185]:
pareto_trials

[FrozenTrial(number=131, state=<TrialState.COMPLETE: 1>, values=[0.6923653843269094, 0.7562557296672298], datetime_start=datetime.datetime(2025, 12, 30, 15, 9, 19, 426980), datetime_complete=datetime.datetime(2025, 12, 30, 15, 9, 19, 751380), params={'top_thr_0': 0.28974521655930124, 'top_thr_1': 0.4062914969308646, 'top_thr_2': 0.2783233287381306, 'top_thr_3': 0.414023301630064, 'top_thr_4': 0.24514448577918485, 'top_thr_5': 0.22910448145450624, 'top_thr_6': 0.17453491223457857, 'top_thr_7': 0.4921051910865488, 'top_thr_8': 0.2006357148621823, 'sub_thr_0': 0.40797545008597025, 'sub_thr_1': 0.18512572877236844, 'sub_thr_2': 0.49827447137649694, 'sub_thr_3': 0.4285797011425786, 'sub_thr_4': 0.28572515039996865, 'sub_thr_5': 0.23329293842415494, 'sub_thr_6': 0.31226930885758647, 'sub_thr_7': 0.22806988384893945, 'sub_thr_8': 0.40348951316978016, 'sub_thr_9': 0.48034368670120264, 'sub_thr_10': 0.4346008910049627, 'sub_thr_11': 0.30943587243762005, 'sub_thr_12': 0.2358596262820377, 'sub_th

In [186]:
len(pareto_trials)

27

In [ ]:
df = study1.trials_dataframe()

df = df[df["state"] == "COMPLETE"]

df = df.rename(columns={
    "values_0": "aspect_precision",
    "values_1": "aspect_recall",
})

best_trial_numbers = [t.number for t in study1.best_trials]

pareto_df = df[df['number'].isin(best_trial_numbers)]

csv_filename = "optuna_results_final.csv"
pareto_df.to_csv(csv_filename, index=False)

print(f" Saved {len(df)} trials to {csv_filename}")

files.download(csv_filename)

✅ Saved 500 trials to optuna_results_final.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [179]:
pareto_df

,number,aspect_precision,aspect_recall,datetime_start,datetime_complete,duration,params_sub_thr_0,params_sub_thr_1,params_sub_thr_10,params_sub_thr_11,...,params_top_thr_0,params_top_thr_1,params_top_thr_2,params_top_thr_3,params_top_thr_4,params_top_thr_5,params_top_thr_6,params_top_thr_7,params_top_thr_8,state
223,223,0.694568,0.754868,2025-12-30 14:59:38.786687,2025-12-30 14:59:39.028840,0 days 00:00:00.242153,0.252676,0.291956,0.337966,0.394588,...,0.546977,0.239746,0.594806,0.296796,0.150702,0.548692,0.241262,0.862657,0.404102,COMPLETE
233,233,0.691001,0.756935,2025-12-30 14:59:41.295537,2025-12-30 14:59:41.542047,0 days 00:00:00.246510,0.252009,0.229187,0.213806,0.312998,...,0.393161,0.190062,0.478213,0.665362,0.163894,0.473309,0.154094,0.818335,0.454980,COMPLETE
241,241,0.687937,0.761863,2025-12-30 14:59:43.312403,2025-12-30 14:59:43.564552,0 days 00:00:00.252149,0.317752,0.272401,0.241423,0.353511,...,0.283174,0.229440,0.827507,0.410184,0.255896,0.227954,0.150328,0.797515,0.403917,COMPLETE
258,258,0.696315,0.751104,2025-12-30 14:59:47.692997,2025-12-30 14:59:48.063142,0 days 00:00:00.370145,0.290881,0.319087,0.318100,0.460796,...,0.597489,0.259982,0.394325,0.346843,0.172750,0.483211,0.387684,0.858361,0.307378,COMPLETE
272,272,0.677046,0.770993,2025-12-30 14:59:52.179703,2025-12-30 14:59:52.450601,0 days 00:00:00.270898,0.406623,0.220984,0.208335,0.453717,...,0.466294,0.416368,0.679213,0.160427,0.150202,0.176803,0.177095,0.483935,0.300761,COMPLETE
303,303,0.686397,0.761919,2025-12-30 15:00:00.211208,2025-12-30 15:00:00.471955,0 days 00:00:00.260747,0.296584,0.243263,0.166493,0.268837,...,0.244449,0.305481,0.889229,0.533355,0.242940,0.428865,0.182029,0.586003,0.258318,COMPLETE
304,304,0.669735,0.776222,2025-12-30 15:00:00.473138,2025-12-30 15:00:00.854264,0 days 00:00:00.381126,0.384936,0.215679,0.168790,0.309418,...,0.321284,0.282140,0.415482,0.330311,0.175639,0.300982,0.298391,0.517929,0.218015,COMPLETE
311,311,0.682209,0.770646,2025-12-30 15:00:03.196613,2025-12-30 15:00:03.512573,0 days 00:00:00.315960,0.305764,0.217004,0.203483,0.238210,...,0.416407,0.166481,0.285239,0.529466,0.164863,0.432747,0.192410,0.673485,0.407154,COMPLETE
315,315,0.661897,0.776684,2025-12-30 15:00:04.321295,2025-12-30 15:00:04.584798,0 days 00:00:00.263503,0.407786,0.198685,0.209607,0.232059,...,0.270001,0.818407,0.313289,0.176929,0.167185,0.350578,0.373394,0.577297,0.248729,COMPLETE
340,340,0.682477,0.765168,2025-12-30 15:00:10.970818,2025-12-30 15:00:11.246111,0 days 00:00:00.275293,0.274465,0.332624,0.180655,0.270430,...,0.447398,0.155182,0.420310,0.541395,0.252241,0.361909,0.224901,0.578246,0.508967,COMPLETE
